In [17]:
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (
    load_breast_cancer,
    fetch_openml
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = str(PROJECT_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import logistic_regression
import optimizers

importlib.reload(logistic_regression)
importlib.reload(optimizers)

RANDOM_STATE = 42

In [18]:
# Dataset 1: Breast Cancer Wisconsin

breast_cancer = load_breast_cancer()

datasets = {
    "Breast Cancer": {
        "X": breast_cancer.data,
        "y": breast_cancer.target
    }
}

# Dataset 2: Banknote Authentication

banknote = fetch_openml(
    name="banknote-authentication",
    version=1,
    as_frame=False,
    parser="auto"
)

datasets["Banknote"] = {
    "X": banknote.data.astype(float),
    "y": pd.factorize(banknote.target)[0]
}

# Dataset 3: Ionosphere

ionosphere = fetch_openml(
    name="ionosphere",
    version=1,
    as_frame=False,
    parser="auto"
)

datasets["Ionosphere"] = {
    "X": ionosphere.data.astype(float),
    "y": pd.factorize(ionosphere.target)[0]
}

In [19]:
dataset_summary = []

for name, dataset in datasets.items():

    X_data = dataset["X"]
    y_data = dataset["y"]

    dataset_summary.append({
        "Dataset": name,
        "Samples": X_data.shape[0],
        "Features": X_data.shape[1],
        "Classes": len(np.unique(y_data)),
        "Class 0": np.sum(y_data == 0),
        "Class 1": np.sum(y_data == 1),
        "Missing Values": np.isnan(X_data).sum()
    })

summary_df = pd.DataFrame(dataset_summary)

display(summary_df)

,Dataset,Samples,Features,Classes,Class 0,Class 1,Missing Values
0,Breast Cancer,569,30,2,212,357,0
1,Banknote,1372,4,2,762,610,0
2,Ionosphere,351,34,2,225,126,0


In [20]:
prepared_datasets = {}

for name, dataset in datasets.items():

    X_data = dataset["X"]
    y_data = dataset["y"]

    # Split before standardization to prevent data leakage
    X_train, X_test, y_train, y_test = train_test_split(
        X_data,
        y_data,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_data
    )

    # Fit scaler only on training data
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    prepared_datasets[name] = {
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler
    }

    print(f"\n{name}")
    print("Training samples:", len(y_train))
    print("Testing samples:", len(y_test))
    print("Training feature means:", np.round(X_train_scaled.mean(axis=0)[:5], 4))
    print("Training feature std:", np.round(X_train_scaled.std(axis=0)[:5], 4))


Breast Cancer
Training samples: 455
Testing samples: 114
Training feature means: [-0.  0. -0.  0.  0.]
Training feature std: [1. 1. 1. 1. 1.]

Banknote
Training samples: 1097
Testing samples: 275
Training feature means: [-0. -0. -0.  0.]
Training feature std: [1. 1. 1. 1.]

Ionosphere
Training samples: 280
Testing samples: 71
Training feature means: [-0.  0.  0.  0. -0.]
Training feature std: [1. 0. 1. 1. 1.]


In [21]:
for name, data in prepared_datasets.items():

    X_train = data["X_train"]

    constant_features = np.where(
        np.std(X_train, axis=0) < 1e-10
    )[0]

    print(f"\n{name}")
    print("Number of constant features:", len(constant_features))
    print("Constant feature indices:", constant_features)


Breast Cancer
Number of constant features: 0
Constant feature indices: []

Banknote
Number of constant features: 0
Constant feature indices: []

Ionosphere
Number of constant features: 1
Constant feature indices: [1]


In [22]:
# Remove constant features from each prepared dataset

for name, data in prepared_datasets.items():

    X_train = data["X_train"]
    X_test = data["X_test"]

    # Identify non-constant features using training data only
    feature_std = np.std(X_train, axis=0)
    keep_mask = feature_std > 1e-10

    removed_features = np.where(~keep_mask)[0]

    # Apply identical feature selection to train and test
    data["X_train"] = X_train[:, keep_mask]
    data["X_test"] = X_test[:, keep_mask]
    data["feature_mask"] = keep_mask

    print(f"\n{name}")
    print("Removed feature indices:", removed_features)
    print("Remaining features:", data["X_train"].shape[1])


Breast Cancer
Removed feature indices: []
Remaining features: 30

Banknote
Removed feature indices: []
Remaining features: 4

Ionosphere
Removed feature indices: [1]
Remaining features: 33


In [23]:
hessian_results = []

for name, data in prepared_datasets.items():

    X_train = data["X_train"]

    # Add intercept
    X_bias = logistic_regression.add_intercept(X_train)

    # Initialize parameters to zero
    theta = np.zeros(X_bias.shape[1])

    # Calculate Hessian
    H = logistic_regression.compute_hessian(
        X_bias,
        theta,
        l2=0.0
    )

    # Calculate eigenvalues
    eigenvalues = np.linalg.eigvalsh(H)

    # Numerical tolerance for positive eigenvalues
    tolerance = 1e-10
    positive_eigenvalues = eigenvalues[eigenvalues > tolerance]

    if len(positive_eigenvalues) < len(eigenvalues):
        condition_number = np.inf
    else:
        condition_number = (
            positive_eigenvalues[-1] /
            positive_eigenvalues[0]
        )

    hessian_results.append({
        "Dataset": name,
        "Minimum Eigenvalue": eigenvalues[0],
        "Maximum Eigenvalue": eigenvalues[-1],
        "Condition Number": condition_number
    })

    print(f"\n{name}")
    print("Eigenvalues:", np.round(eigenvalues, 8))
    print("Condition Number:", condition_number)

hessian_df = pd.DataFrame(hessian_results)

display(hessian_df)


Breast Cancer
Eigenvalues: [3.20800000e-05 1.93010000e-04 4.28800000e-04 1.62077000e-03
 1.84131000e-03 3.78490000e-03 4.41249000e-03 5.39834000e-03
 6.49816000e-03 6.94277000e-03 7.77536000e-03 1.11825500e-02
 1.40022600e-02 1.52944300e-02 1.92859600e-02 2.34187700e-02
 3.98029700e-02 5.99810000e-02 6.35571700e-02 6.94701900e-02
 8.59081900e-02 9.59087500e-02 1.18748790e-01 1.63632440e-01
 2.50000000e-01 2.95090070e-01 4.13826780e-01 5.04351710e-01
 7.15751720e-01 1.42084637e+00 3.33101188e+00]
Condition Number: 103824.43345038652

Banknote
Eigenvalues: [0.04273372 0.08819869 0.25       0.32715148 0.54191611]
Condition Number: 12.681228633785313

Ionosphere
Eigenvalues: [0.01452817 0.02267414 0.02410973 0.02561431 0.03209198 0.0385571
 0.03884552 0.04822629 0.05411254 0.0547543  0.06743692 0.07278965
 0.07799593 0.09267596 0.09740271 0.10039585 0.10524164 0.11671796
 0.12537412 0.14196048 0.14430561 0.17256368 0.19234174 0.21072814
 0.23872835 0.25       0.25541484 0.30509439 0.32563

,Dataset,Minimum Eigenvalue,Maximum Eigenvalue,Condition Number
0,Breast Cancer,0.000032,3.331012,103824.433450
1,Banknote,0.042734,0.541916,12.681229
2,Ionosphere,0.014528,2.228349,153.381278


In [25]:
ablation_datasets = {}

for name, dataset in datasets.items():

    X_data = dataset["X"]
    y_data = dataset["y"]

    X_train, X_test, y_train, y_test = train_test_split(
        X_data,
        y_data,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_data
    )

    # Remove constant features based on training data
    feature_std = np.std(X_train, axis=0)
    keep_mask = feature_std > 1e-10

    X_train = X_train[:, keep_mask]
    X_test = X_test[:, keep_mask]

    # Standardize using training data only
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    ablation_datasets[name] = {
        "original": (X_train, X_test),
        "standardized": (X_train_scaled, X_test_scaled),
        "y_train": y_train,
        "y_test": y_test,
        "feature_mask": keep_mask
    }

In [26]:
ablation_conditioning = []

for name, data in ablation_datasets.items():

    for condition in ["original", "standardized"]:

        X_train, _ = data[condition]

        X_bias = logistic_regression.add_intercept(X_train)
        theta = np.zeros(X_bias.shape[1])

        H = logistic_regression.compute_hessian(
            X_bias,
            theta,
            l2=0.0
        )

        eigenvalues = np.linalg.eigvalsh(H)

        tolerance = 1e-10
        positive_eigenvalues = eigenvalues[eigenvalues > tolerance]

        if len(positive_eigenvalues) < len(eigenvalues):
            condition_number = np.inf
        else:
            condition_number = (
                positive_eigenvalues[-1] /
                positive_eigenvalues[0]
            )

        ablation_conditioning.append({
            "Dataset": name,
            "Condition": condition,
            "Minimum Eigenvalue": eigenvalues[0],
            "Maximum Eigenvalue": eigenvalues[-1],
            "Condition Number": condition_number
        })

ablation_conditioning_df = pd.DataFrame(ablation_conditioning)

display(ablation_conditioning_df.round(6))

,Dataset,Condition,Minimum Eigenvalue,Maximum Eigenvalue,Condition Number
0,Breast Cancer,original,0.000000,403396.261110,2.184378e+12
1,Breast Cancer,standardized,0.000032,3.331012,1.038244e+05
2,Banknote,original,0.097829,12.917883,1.320454e+02
3,Banknote,standardized,0.042734,0.541916,1.268123e+01
4,Ionosphere,original,0.004300,1.704614,3.963977e+02
5,Ionosphere,standardized,0.014528,2.228349,1.533813e+02


In [27]:
improvement_results = []

for name in ablation_datasets:

    original = ablation_conditioning_df[
        (ablation_conditioning_df["Dataset"] == name) &
        (ablation_conditioning_df["Condition"] == "original")
    ]["Condition Number"].iloc[0]

    standardized = ablation_conditioning_df[
        (ablation_conditioning_df["Dataset"] == name) &
        (ablation_conditioning_df["Condition"] == "standardized")
    ]["Condition Number"].iloc[0]

    reduction = (1 - standardized / original) * 100

    improvement_results.append({
        "Dataset": name,
        "Original Condition Number": original,
        "Standardized Condition Number": standardized,
        "Reduction (%)": reduction
    })

improvement_df = pd.DataFrame(improvement_results)

display(improvement_df.round(4))

,Dataset,Original Condition Number,Standardized Condition Number,Reduction (%)
0,Breast Cancer,2.184378e+12,103824.4335,100.0000
1,Banknote,1.320454e+02,12.6812,90.3963
2,Ionosphere,3.963977e+02,153.3813,61.3062
